# Lab 8: Generating movie reviews

**DSAN-6600 Deep Learning · Fall 2026**

**Due:** Wednesday Oct 28, 11:59 PM ET · **Target time:** about 75 minutes

---

## What this lab is for

The old version of this lab trained a decoder-only transformer on movie reviews
for about two hours. That training is already done. You download the weights
and use them.

A review is a sequence, and the training task is to predict the next word.
You will build that task in four steps:

1. **The target.** Pair each word with the word that follows it.
2. **The causal mask.** Stop a position from seeing the future, and check that
   the weights still sum to 1.
3. **One head of attention.** Compute the weighted blend of values, and see
   that a later word gets weight 0.
4. **Generation.** Load the trained model and append words it samples, one at
   a time.

## How this lab is graded

- **The notebook is graded for completion, not for correctness.** Submit it,
  with all cells run and outputs visible, and you get the points.
- **The sample is seeded, so it should match a classmate's.** A mismatch is a
  bug, or a cell that was run out of order. Run All from the top if that
  happens.
- **The understanding is assessed on Quiz 9**, the Thursday after the due date.
- **Solutions are posted after the deadline.**

## Using AI on this lab

Allowed and expected, with disclosure; there is a cell for that at the end.
An agent will write the mask in one shot. It is much less likely to tell you
why a row of that mask still sums to 1, and Part 5 asks you to.

## Step 0: Setup

::: {.callout-note}
## No GPU required
The weights are about 105 MB. Inference on the sentences in this lab is a
second or two on CPU. A GPU runtime does no harm.
:::

In [ ]:
import json
import os
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 6600
torch.manual_seed(SEED)

PAD_WORD, UNK_WORD = "PAD", "UNK"
SEQ = 80                 # the trained model always sees 80 positions
EMB, HEADS, BLOCKS = 512, 4, 2
FF = 1028

WEIGHT_URL = ("https://benjaminhoughton.georgetown.domains/deep-learning/"
              "labs/lab-8/mini-gpt.pt")
VOCAB_URL = ("https://benjaminhoughton.georgetown.domains/deep-learning/"
             "labs/lab-8/vocab.json")


def fetch(url, name, env):
    override = os.environ.get(env)
    if override and Path(override).exists():
        return Path(override)
    dest = Path(name)
    if not dest.exists() or dest.stat().st_size < 1000:
        print(f"downloading {name}")
        urllib.request.urlretrieve(url, dest)
    return dest


weights_path = fetch(WEIGHT_URL, "mini-gpt.pt", "LAB8_WEIGHTS")
vocab_path = fetch(VOCAB_URL, "vocab.json", "LAB8_VOCAB")
id_to_word = json.loads(vocab_path.read_text())
word_to_id = {w: i for i, w in enumerate(id_to_word)}

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print(f"torch {torch.__version__}")
print(f"vocabulary {len(id_to_word):,} tokens, "
      f"weights {weights_path.stat().st_size/1e6:.0f} MB")

---

# Part 1 · The target is the next word

The vocabulary is the one the trained model uses. `PAD` is 0 and `UNK` is 1.
Turn one sentence into the pairs that model was trained on.

In [ ]:
def encode(tokens):
    return [word_to_id.get(w, word_to_id[UNK_WORD]) for w in tokens]


SENTENCE = ("this movie is a very entertaining movie but it is not "
            "the best film i have ever seen")
print(SENTENCE)
print("ids:", encode(SENTENCE.split()))

## 1a · Shift by one

Given the ids of one review, the input is every token except the last and the
target is every token except the first. Position $t$ of the target is the word
that follows position $t$ of the input.

In [ ]:
def inputs_and_targets(ids):
    """ids: 1D LongTensor. Return x, y with y the next token at each position."""
    # TODO: drop the last id from x and the first id from y.
    #   x and y have the same length, one shorter than ids.
    raise NotImplementedError("inputs_and_targets")


demo_ids = torch.tensor(encode(SENTENCE.split()))
x, y = inputs_and_targets(demo_ids)
print(f"{'input':<16} {'target'}")
for a, b in zip(x.tolist(), y.tolist()):
    print(f"{id_to_word[a]:<16} {id_to_word[b]}")
print(f"\nlengths: ids {len(demo_ids)}, x {len(x)}, y {len(y)}")

That table is the whole training task. The model sees the left column and is
scored on the right column. At generation time the right column does not exist
yet, which is Part 4.

---

# Part 2 · The causal mask

Position $t$ is trying to predict the word at $t+1$. If it can see that word,
the task is trivial. The causal mask puts $-\infty$ on every future position
so the softmax gives it weight 0.

In [ ]:
def causal_mask(T):
    """(T, T) float tensor. 0 on and below the diagonal, -inf strictly above."""
    # TODO: build the mask. torch.triu with diagonal=1 is the upper triangle.
    raise NotImplementedError("causal_mask")


T = 6
mask = causal_mask(T)
torch.manual_seed(SEED)
scores = torch.randn(T, T)
weights = torch.softmax(scores + mask, dim=-1)

print("mask, row 0:", mask[0].tolist())
print(f"after softmax, row 0 sums to {weights[0].sum():.2f} "
      f"and puts {weights[0, 0]:.2f} on itself")
print(f"weight on any future position: {weights.triu(1).abs().max():.1f}")
print(f"every row sums to 1: "
      f"{torch.allclose(weights.sum(-1), torch.ones(T))}")

The first position has nothing before it, so its whole weight lands on itself.
Later positions spread their weight over the past. Nothing lands on the future,
and the row still sums to 1 because the softmax runs after the mask.

---

# Part 3 · One head of attention

Attention turns those weights into a mix of value vectors. The scale
$\sqrt{d}$ keeps the dot products from saturating the softmax, which is the
same reason as in the lecture.

In [ ]:
def scaled_attention(Q, K, V, mask):
    """Q, K, V are (T, d). Return (output, weights), output shaped like V."""
    # TODO:
    #   scores = Q @ K transposed, divided by sqrt(d)
    #   add mask, softmax over the last dimension, multiply by V
    raise NotImplementedError("scaled_attention")


words = "the movie was not very good".split()
torch.manual_seed(SEED)
d = 8
Q = torch.randn(len(words), d)
K = torch.randn(len(words), d)
V = torch.randn(len(words), d)
output, attn = scaled_attention(Q, K, V, causal_mask(len(words)))

print(f"{'':<10}" + "".join(f"{w:>8}" for w in words))
for i, w in enumerate(words):
    row = " ".join(f"{attn[i, j]:8.2f}" for j in range(len(words)))
    print(f"{w:<10}{row}")
print(f"\noutput shape {tuple(output.shape)}")
print(f"'good' puts {attn[-1, -1]:.2f} on itself and "
      f"{attn[-1, :].sum():.2f} on the whole row")

Read across a row. Weight to the right of the diagonal is 0, because that row's
mask put $-\infty$ there. The output at each word is a blend of the value
vectors at the words it was allowed to see.

## 3b · More heads do not make the vector wider

`embed_dim` has to be divisible by `num_heads` because each head gets an equal
slice of the same vector. Adding heads makes the slices thinner. It does not
add dimensions, and it does not add parameters in the projections.

In [ ]:
def attention_params(nhead):
    layer = nn.MultiheadAttention(64, nhead, batch_first=True)
    return sum(p.numel() for p in layer.parameters())


for h in (1, 4, 8):
    print(f"  {h} head{'s' if h > 1 else ' '}: {attention_params(h):,} parameters, "
          f"each head is {64 // h} wide")

Same parameter count at 1, 4 and 8 heads. The capacity of one attention layer
is set by the width, not by the head count. Heads are there so different slices
can learn different patterns, which is the multi-head slide.

---

# Part 4 · Load the trained model, then generate

The 2024 run used two blocks, width 512, 4 heads, and a vocabulary of 20,000.
It trained for 200 epochs, about two hours. The file you downloaded is that
model. The class below is the architecture those weights were saved from, so
the names have to match. You do not need to modify it.

Training showed the model the true past and scored the true next word, one
forward pass for the whole sequence. Generation does not have the true next
word. It samples a token, appends it, and runs again.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, ff_dim, num_heads, dropout, norm_eps):
        super().__init__()
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.attention = nn.MultiheadAttention(
            embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
        self.drop_out_1 = nn.Dropout(dropout)
        self.layer_norm_1 = nn.LayerNorm(embed_dim, eps=norm_eps)
        self.linear_stack = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim),
            nn.ReLU(),
        )
        self.drop_out_2 = nn.Dropout(dropout)
        self.layer_norm_2 = nn.LayerNorm(embed_dim, eps=norm_eps)

    def forward(self, x):
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        mask = nn.Transformer.generate_square_subsequent_mask(
            q.shape[-2], device=x.device)
        attention, _ = self.attention(
            q, k, v, need_weights=False, attn_mask=mask, is_causal=True)
        attention = self.layer_norm_1(x + self.drop_out_1(attention))
        forward = self.drop_out_2(self.linear_stack(attention))
        return self.layer_norm_2(attention + forward)


class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.position = torch.arange(SEQ)
        self.word_embedding = nn.Embedding(len(id_to_word), EMB, padding_idx=0)
        self.position_embedding = nn.Embedding(SEQ, EMB)
        self.blocks = nn.ModuleList([
            TransformerBlock(EMB, FF, HEADS, dropout=0.1, norm_eps=1e-6)
            for _ in range(BLOCKS)])
        self.linear = nn.Linear(EMB, len(id_to_word))

    def forward(self, x):
        pos = self.position.to(x.device)
        h = self.word_embedding(x) + self.position_embedding(pos)
        for block in self.blocks:
            h = block(h)
        return self.linear(h)


ckpt = torch.load(weights_path, map_location="cpu", weights_only=False)
model = Decoder()
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"loaded epoch {ckpt['epoch']}, "
      f"training loss {ckpt['train_loss'][0]:.2f} -> {ckpt['train_loss'][-1]:.2f}")

plt.figure(figsize=(6, 3.2))
plt.plot(ckpt["train_loss"], color="#2980b9")
plt.xlabel("epoch")
plt.ylabel("training loss")
plt.title("The two-hour run you are not repeating")
plt.tight_layout()
plt.show()

Loss fell from about 10 to about 2.3. The next cell scores the loaded model
the way training did: show the true past, ask for the true next word. `PAD`
positions are not scored.

In [ ]:
words = (SENTENCE + " and the acting is fine but the plot is thin").split()
ids = encode(words) + [0] * (SEQ + 1 - len(words))
ids = torch.tensor(ids)
x, y = inputs_and_targets(ids)
y = y.clone()
y[y == 0] = -100
with torch.no_grad():
    pred = model(x[None])[0].argmax(dim=-1)
known = y != -100
acc = (pred[known] == y[known]).float().mean().item()
print(f"teacher-forcing accuracy on this sentence: {acc:.1%}")
print(f"positions scored: {int(known.sum())} of {SEQ}")

## 4a · Append the word it just chose

The model always consumes a length-80 sequence. The prompt occupies the front
and the rest is `PAD`. The logits at the last prompt word are the distribution
for the next word. Take the top 10, sample one, write it into the next slot,
and repeat.

In [ ]:
def generate(model, prompt, steps=24, temperature=1.0, k=10):
    """Sample `steps` new words after `prompt`, a list of words."""
    idx = len(prompt) - 1
    padded = prompt + [PAD_WORD] * (SEQ - len(prompt))
    sequence = np.array([[word_to_id[w] for w in padded]])
    rng = np.random.default_rng(SEED)
    predicted = []
    while len(predicted) < steps:
        # TODO:
        #   logits are model(sequence) at position idx
        #   keep the top k logits and their token ids
        #   softmax, raise to 1/temperature, renormalize
        #   sample one id with rng.choice
        #   append that word, then write the id into sequence at idx + 1
        raise NotImplementedError("generate")
    return prompt + predicted


print(" ".join(generate(model, ["this", "movie", "is"])))

Each new word was drawn from the model, then written into the sequence as if
it had been in the review. One forward pass produced one word. Training scored
every position of the true review in one pass, because those words were already
there and the mask kept each position from seeing the future.

---

# Part 5 · What this lab showed

Six questions. If you can answer them, you are ready for the Lab 8 part of
Quiz 9.

1. **What the target at position $t$ is**, and why that makes the task trivial
   if position $t$ can see the future.
2. **Why a causal mask's rows still sum to 1**, even though some of the scores
   were $-\infty$.
3. **What the output of attention at one word is a blend of**, and which words
   are excluded.
4. **Why 8 heads have the same number of parameters as 1 head** at a fixed
   width.
5. **What teacher-forcing accuracy measures**, and why it is not "did the
   generated review make sense."
6. **Why generation takes one forward pass per new word** and training takes
   one forward pass for the whole sequence.

---

# Part 6 · AI disclosure

Required if you used any generative AI on this lab. Name the tool, say where you
used it, and say what for. "None" is a perfectly good answer.

*Tool(s):*

*Where:*

*What for:*

---

# Submitting

1. **Restart the kernel and Run All.** Confirm it runs top to bottom.
2. Check that the loss plot and the generated sentence are visible.
3. Render to **HTML or PDF** with resources embedded.
4. Submit the rendered file on Canvas. Due **Wed Oct 28, 11:59 PM ET**.

The notebook is graded for completion. Part 5 is the study list for the quiz.

::: {.callout-note}
## Run from the top if a number looks off
The training and the sample both start from seed 6600. Running the training
cell twice without rerunning the seed line continues from weights that were
already trained, and the curve will not match.
:::